# Week 4 Discussion: Loose vs. Tight Tool Schemas

This notebook tests the same support-ticket request against a deliberately loose tool schema and a constrained schema. It uses Gemini through the OpenAI-compatible endpoint, matching the Week 3 course setup. Store `GEMINI_API_KEY` in Colab Secrets or as an environment variable.

**Prompt:** Find the 10 highest-priority open AI-related support tickets from the last 30 days, assigned to the Facilities team.


In [ ]:
!pip -q install openai jsonschema


In [ ]:
import os, sys, json
from jsonschema import Draft202012Validator

MODEL_NAME = 'gemini-3.5-flash-lite'

def get_gemini_key():
    if 'google.colab' in sys.modules:
        from google.colab import userdata
        try:
            return userdata.get('GEMINI_API_KEY')
        except Exception:
            return None
    return os.environ.get('GEMINI_API_KEY')

GEMINI_API_KEY = get_gemini_key()
print('Live API available:', bool(GEMINI_API_KEY))


In [ ]:
PROMPT = 'Find the 10 highest-priority open AI-related support tickets from the last 30 days, assigned to the Facilities team.'

loose_schema = {
    'type': 'object',
    'properties': {
        'status': {'type': 'string'},
        'priority': {'type': 'string'},
        'team': {'type': 'string'},
        'topic': {'type': 'string'},
        'days_back': {'type': 'number'},
        'limit': {'type': 'number'},
    },
}

tight_schema = {
    'type': 'object',
    'properties': {
        'status': {'type': 'string', 'enum': ['open', 'closed', 'pending']},
        'priority': {'type': 'string', 'enum': ['low', 'medium', 'high', 'critical']},
        'team': {'type': 'string', 'enum': ['Facilities', 'Sales', 'Support']},
        'topic': {'type': 'string', 'enum': ['AI', 'billing', 'software', 'hardware']},
        'days_back': {'type': 'integer', 'minimum': 1, 'maximum': 365},
        'limit': {'type': 'integer', 'minimum': 1, 'maximum': 100},
    },
    'required': ['status', 'priority', 'team', 'topic', 'days_back', 'limit'],
    'additionalProperties': False,
}

print('Loose schema:')
print(json.dumps(loose_schema, indent=2))
print('\nTight schema:')
print(json.dumps(tight_schema, indent=2))


In [ ]:
def call_tool(schema):
    if not GEMINI_API_KEY:
        raise RuntimeError('Set GEMINI_API_KEY before running the live LLM test.')

    from openai import OpenAI
    client = OpenAI(
        api_key=GEMINI_API_KEY,
        base_url='https://generativelanguage.googleapis.com/v1beta/openai/'
    )

    tools = [{
        'type': 'function',
        'function': {
            'name': 'search_support_tickets',
            'description': 'Search support tickets using structured filters.',
            'parameters': schema,
        },
    }]

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{'role': 'user', 'content': PROMPT}],
        tools=tools,
        tool_choice={'type': 'function', 'function': {'name': 'search_support_tickets'}},
        temperature=0,
    )

    call = response.choices[0].message.tool_calls[0]
    return json.loads(call.function.arguments)

def validate(instance, schema):
    errors = sorted(Draft202012Validator(schema).iter_errors(instance), key=lambda e: list(e.path))
    return [e.message for e in errors]


## Test 1: Loose schema


In [ ]:
if GEMINI_API_KEY:
    loose_output = call_tool(loose_schema)
else:
    # Captured example from the initial discussion test; rerun live before posting.
    loose_output = {
        'status': 'currently_open',
        'priority': 'highest',
        'team': 'Facilities',
        'topic': 'artificial intelligence',
        'days_back': 30,
        'limit': 10,
    }

print(json.dumps(loose_output, indent=2))
print('Valid against loose schema:', not validate(loose_output, loose_schema))
print('Errors if evaluated against tight schema:')
for err in validate(loose_output, tight_schema):
    print('-', err)


The loose schema accepts strings that are syntactically correct but may not exist in the database vocabulary. It also does not require fields, cap numeric ranges, or reject unexpected properties.


## Test 2: Tight schema


In [ ]:
if GEMINI_API_KEY:
    tight_output = call_tool(tight_schema)
else:
    # Captured example from the initial discussion test; rerun live before posting.
    tight_output = {
        'status': 'open',
        'priority': 'critical',
        'team': 'Facilities',
        'topic': 'AI',
        'days_back': 30,
        'limit': 10,
    }

print(json.dumps(tight_output, indent=2))
tight_errors = validate(tight_output, tight_schema)
print('Valid against tight schema:', not tight_errors)
for err in tight_errors:
    print('-', err)


## Remaining semantic issue

Even when the tight call is structurally valid, `priority = critical` may not be the best interpretation of **highest-priority**. The user might mean “return all matching open AI tickets sorted from highest to lowest priority.” A later schema revision could add explicit `sort_by` and `sort_order` parameters instead of turning the phrase into a priority filter.


In [ ]:
# Edge-case validation tests that show what the tight schema rejects.
edge_cases = {
    'negative days': {**tight_output, 'days_back': -30},
    'fractional limit': {**tight_output, 'limit': 10.5},
    'invented status': {**tight_output, 'status': 'currently_open'},
    'extra field': {**tight_output, 'sort': 'descending'},
}

for name, case in edge_cases.items():
    errors = validate(case, tight_schema)
    print(f'{name}:', 'REJECTED' if errors else 'ACCEPTED')
    for err in errors:
        print('  -', err)


## Conclusion

Schema constraints reduce the space of invalid tool calls, but they do not eliminate ambiguity in natural-language intent. The most useful constraints are those tied to actual application invariants: allowed values, required parameters, types, numeric ranges, and whether extra fields are permitted.
